# 04 · Urban Flood Risk Analysis

Multiple rainfall scenarios × road network.

> **Prerequisite:** `00_setup.ipynb`

In [ ]:
import sys
if '/content/geo_sp_project/src' not in sys.path:
    sys.path.extend(['/content/geo_sp_project/src', '/content/geo_sp_project/configs'])
    import config; from geo_sp.auth import init_ee; init_ee(config.EE_PROJECT)
!pip install -q osmnx scipy


In [ ]:
import config
from geo_sp.boundaries import get_lga
from geo_sp.terrain import load_terrain_layers
from geo_sp.urban_flood import (
    fetch_drainage_pits, compute_depression_depth, compute_flow_accumulation,
    fetch_roads_osmnx, sample_terrain_at_roads, add_drain_distance,
    run_all_scenarios, aggregate_to_roads, comparison_table,
    plot_scenarios, build_flood_map
)

randwick = get_lga(config.LGA_NAME, config.LGA_LAYER)
layers   = load_terrain_layers()
print('Boundary + terrain loaded')


In [ ]:
# ── Drainage pits ─────────────────────────────────────────────
gdf_drains = fetch_drainage_pits(randwick)
print(f'Drainage pits: {len(gdf_drains) if gdf_drains is not None else 0}')


In [ ]:
# ── Road network ──────────────────────────────────────────────
gdf_roads = fetch_roads_osmnx(randwick)
print(f'Roads: {len(gdf_roads)}')


In [ ]:
# ── Terrain at road points ────────────────────────────────────
depression = compute_depression_depth(layers['elevation'], randwick)
flow_acc   = compute_flow_accumulation(layers['elevation'], randwick)

gdf_pts = sample_terrain_at_roads(gdf_roads, layers['elevation'], flow_acc, depression)
gdf_pts = add_drain_distance(gdf_pts, gdf_drains)
print(f'Road sample points: {len(gdf_pts)}')


In [ ]:
# ── Risk for all scenarios ────────────────────────────────────
results = run_all_scenarios(gdf_pts, config.RAINFALL_SCENARIOS)
agg     = aggregate_to_roads(results, gdf_roads)
df_cmp  = comparison_table(agg, config.RAINFALL_SCENARIOS)
print(df_cmp.to_string(index=False))


In [ ]:
# ── Plots ─────────────────────────────────────────────────────
plot_scenarios(df_cmp)


In [ ]:
# ── Interactive map (Extreme scenario) ───────────────────────
extreme_key = 'Extreme (100mm/hr)'
if extreme_key in agg:
    m = build_flood_map(agg[extreme_key], gdf_drains, extreme_key,
                        config.RAINFALL_SCENARIOS[extreme_key], config.MAP_CENTER)
    m.save('flood_map_extreme.html')
    from IPython.display import display; display(m)

# Save CSVs
df_cmp.to_csv('rainfall_scenario_comparison.csv', index=False)
print('Saved: rainfall_scenario_comparison.csv')
